In [ ]:
suppressMessages(suppressWarnings({
library(SeuratDisk)
library(tidyverse)
library(Seurat)
}))

In [ ]:
subproj <- "AVGUSTINOVA_15"
source(here::here(glue::glue("{subproj}/st_analysis/misc/paths.R")))

df_info <- read_csv(here::here(glue::glue("utils/info_samples.csv")),show_col_types = FALSE)
df_info <- df_info[df_info["subproject"] == subproj, ]
df_info <- df_info[df_info["quality"] != "Bad", ]

In [ ]:
py_path <- here::here(glue::glue("{clust}/{ver}/py_objects_{ver}"))
r_path <- here::here(glue::glue("{ann}/{robj_dir}"))

In [ ]:
for (pat in df_info$patient) {
    print(pat)
    counts <- as.data.frame(read_csv(glue::glue(py_path,"/counts_matrix_{pat}.csv"),show_col_types = FALSE))
    
    rownames(counts) <- counts[[1]]
    counts <- counts[,-1]
    counts <- as.matrix(counts)
    
    metadata <- as.data.frame(read_csv(glue::glue(py_path,"/metadata_{pat}.csv"),show_col_types = FALSE))
    rownames(metadata) <- metadata[[1]]
    metadata <- metadata[,-1]
    
    seurat_obj <- CreateSeuratObject(counts = t(counts), meta.data = metadata, assay="Spatial")
    
    ## add image
    seurat_obj2 <- readRDS(here::here(glue::glue("{qc}/{robj_dir}/2.2-qc_se_{pat}.rds")))

    seurat_obj2 <- subset(seurat_obj2, cells = colnames(seurat_obj))

    seurat_obj@images <- seurat_obj2@images
    
    saveRDS(seurat_obj, file = here::here(glue::glue("{r_path}/{pat}.rds")))
}